# ForestWatch Papua — Improve Model (Fine-tuning Kelas Minoritas)

**Tujuan**: naikkan IoU 5 kelas lemah (Lahan Terbuka, Sawit, Pertanian Lain, Tambang,
Permukiman) **tanpa menurunkan** 2 kelas mayoritas (Perairan, Hutan), di atas checkpoint
**model_1 (Attention U-Net)** yang sudah dilatih — **tanpa train ulang dari nol**.

**Metode (decoupled fine-tuning — Kang dkk. ICLR 2020, arXiv 1910.09217):**
- Encoder ResNet50 **DIBEKUKAN** (fitur ImageNet paling generalizable) → latih ulang
  **decoder + segmentation head** saja (cukup utk perbaiki separabilitas kelas yang sering
  tertukar: Tambang↔Lahan Terbuka, Sawit↔Pertanian Lain).
- Loss `0.4·Focal + 0.4·Tversky(β>α) + 0.2·CE(weighted)`: Tversky β=0,7>α=0,3 menekan
  false-negative → recall minoritas naik (Abraham & Khan 2019); komponen CE berbobot supaya
  **boost kelas minoritas benar-benar berpengaruh di loss** (preset `focal_tversky` murni
  TIDAK memakai class_weights). Plus `WeightedRandomSampler` (oversample patch kelas langka).
- **Seleksi checkpoint terjaga (guarded)**: tiap epoch, model hanya disimpan bila IoU kelas
  mayoritas (Perairan, Hutan) di **VAL** tidak turun > ε dari baseline → model final dijamin
  tak mengorbankan kelas mayoritas.
- **Anti-bocor**: seleksi pakai VAL; TEST hanya sekali di akhir untuk laporan jujur.
- **Reversible**: hasil → `best_model_finetune.pt` terpisah; `best_model.pt` baseline tak disentuh.

**Cara pakai**: jalankan cell berurutan dari atas. Cell fine-tune aman di-resume bila sesi mati.

**Referensi**: Kang dkk. ICLR 2020 (decoupling/cRT); Oktay dkk. MICCAI 2018, arXiv 1804.03999
(attention-gate, dasar pilih model_1); Abraham & Khan 2019 (focal-Tversky); Eigen & Fergus
2015 (median-frequency).


## Bagian 0 — Setup environment (Colab / Lab / Kaggle)


In [ ]:
# === Bagian 0 — Setup (set ENV = "colab" / "lab" / "kaggle") ===
ENV = "colab"   # "colab" | "lab" (PC+Drive Desktop) | "kaggle"
import os, sys, subprocess, importlib
from pathlib import Path

# Auto-deteksi Kaggle (folder /kaggle hanya ada di runtime Kaggle).
if Path("/kaggle").exists() and ENV != "kaggle":
    print(f"[auto-detect] /kaggle -> override ENV='{ENV}' -> 'kaggle'")
    ENV = "kaggle"

DRIVE_ROOT = None
_SRC = None
if ENV == "colab":
    subprocess.run("cd /content && (git -C fw_repo pull -q || git clone --depth 1 "
                   "https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo)",
                   shell=True, check=False)
    subprocess.run("pip install -q -e /content/fw_repo[ml]", shell=True, check=False)
    _SRC = "/content/fw_repo/model/src"
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/Satria Data 3.0")
elif ENV == "lab":
    DRIVE_ROOT = Path(r"G:/My Drive/Satria Data 3.0")   # <- SESUAIKAN mount Drive Desktop lab
elif ENV == "kaggle":
    subprocess.run("cd /kaggle/working && (git -C fw_repo pull -q || git clone --depth 1 "
                   "https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo)",
                   shell=True, check=False)
    subprocess.run("pip install -q --no-deps -e /kaggle/working/fw_repo[ml]", shell=True, check=False)
    subprocess.run("pip install -q --no-deps segmentation-models-pytorch albumentations torchmetrics",
                   shell=True, check=False)
    _SRC = "/kaggle/working/fw_repo/model/src"
else:
    raise ValueError("ENV harus 'colab' | 'lab' | 'kaggle'")

if _SRC and _SRC not in sys.path:
    sys.path.insert(0, _SRC)
for _m in [m for m in list(sys.modules) if m == "forestwatch" or m.startswith("forestwatch.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

import torch
_gpu = f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""
print(f"ENV={ENV} | CUDA={torch.cuda.is_available()}{_gpu}")


In [ ]:
# === Deklarasi path, identitas model, & config ===
from forestwatch.config import load_config
from forestwatch.utils.io import save_json, load_json
from forestwatch.constants import N_CLASSES, CLASS_NAMES, CLASS_COLORS
cfg = load_config()

MODEL_KEY  = "model_1_attention_unet"   # pemenang banding (compare_and_select_best_model.ipynb)
MODEL_ARCH = dict(architecture="unet_scse", encoder_name="resnet50")
STRONG = [0, 1]            # Perairan, Hutan -- JAGA (jangan turun)
WEAK   = [2, 3, 4, 5, 6]   # Lahan Terbuka, Sawit, Pertanian Lain, Tambang, Permukiman -- target naik

if ENV in ("colab", "lab"):
    BAHAN_DIR   = DRIVE_ROOT / "Bahan_Training_Fix"
    MODELS_ROOT = DRIVE_ROOT / "ForestWatch_Outputs" / "Model_Comparison"
    BASE_DIR    = MODELS_ROOT / MODEL_KEY
    BASE_CKPT   = BASE_DIR / "best_model.pt"
    FT_DIR      = MODELS_ROOT / (MODEL_KEY + "_finetune")
else:  # kaggle: data + checkpoint baseline dari dataset yg di-attach (Add Input)
    BAHAN_DIR = None                       # diisi di cell ekstrak (rglob)
    BASE_DIR  = None
    BASE_CKPT = next(Path("/kaggle/input").rglob("best_model.pt"))
    FT_DIR    = Path("/kaggle/working") / (MODEL_KEY + "_finetune")

FT_DIR.mkdir(parents=True, exist_ok=True)
FT_CKPT    = FT_DIR / "best_model_finetune.pt"
FT_RESUME  = FT_DIR / "best_model_finetune_resume.pt"
FT_SAMPLER = FT_DIR / "patch_sampler_weights_finetune.json"   # cache TERPISAH (wajib, lihat cell build)
OUT_DIR    = FT_DIR

assert BASE_CKPT.exists(), f"Baseline checkpoint tak ada: {BASE_CKPT}"
print("Baseline ckpt :", BASE_CKPT)
print("Output FT dir :", FT_DIR)
print("Kelas JAGA    :", [CLASS_NAMES[c] for c in STRONG])
print("Kelas target  :", [CLASS_NAMES[c] for c in WEAK])


In [ ]:
# === Ekstrak dataset ke disk lokal + daftar file (train/val/test) ===
# Pola sama spt notebook training: extract .tar -> disk lokal (lepas dari bottleneck Drive FUSE).
from forestwatch.data.dataset import extract_dataset_archives
from forestwatch.data.patches import list_patches

if ENV in ("colab", "lab"):
    LOCAL_DIR = Path("/content/dataset_local") if ENV == "colab" else Path.home() / "dataset_local"
    _splits = ("train", "val", "test")
    if (BAHAN_DIR / "train_rajaampat").exists():
        _splits = _splits + ("train_rajaampat",)
    local_dirs = extract_dataset_archives(BAHAN_DIR, LOCAL_DIR, splits=_splits, max_workers=8)
    cw_path = BAHAN_DIR / "class_weights.json"
else:  # kaggle
    BAHAN_SRC = Path("/kaggle/temp/bahan_src"); BAHAN_SRC.mkdir(parents=True, exist_ok=True)
    for _t in ["train", "val", "test", "train_rajaampat", "class_weights.json"]:
        _f = list(Path("/kaggle/input").rglob(_t))
        if _f and not (BAHAN_SRC / _t).exists():
            os.symlink(_f[0], BAHAN_SRC / _t)
    _splits = tuple(s for s in ("train", "val", "test", "train_rajaampat") if (BAHAN_SRC / s).exists())
    if (BAHAN_SRC / "train").is_dir():
        local_dirs = {s: BAHAN_SRC / s for s in _splits}
    else:
        local_dirs = extract_dataset_archives(BAHAN_SRC, Path("/kaggle/temp/dataset_local"),
                                               splits=_splits, max_workers=8)
    cw_path = BAHAN_SRC / "class_weights.json"

final_train_files = list_patches(local_dirs["train"])
if "train_rajaampat" in local_dirs:
    _ra = list_patches(local_dirs["train_rajaampat"])
    final_train_files = final_train_files + _ra
    print(f"  + {len(_ra)} patch Raja Ampat (Tambang asli)")
val_p  = list_patches(local_dirs["val"])
test_p = list_patches(local_dirs["test"])
assert final_train_files and val_p and test_p, "train/val/test kosong -- cek BAHAN_DIR / attach dataset."
print(f"train={len(final_train_files)} | val={len(val_p)} | test={len(test_p)}")


In [ ]:
# === Bobot kelas: median-frequency (baseline) -> boost kelas minoritas (idx 2..6) ===
# class_weights dipakai DUA tempat: (1) komponen CE di loss (lihat cell build), (2) bobot
# oversampling WeightedRandomSampler. Boosting kelas WEAK menaikkan fokus loss + frekuensi
# sampling patch langka. Basis: median-frequency (Eigen & Fergus 2015) + class-balanced (Kang 2020).
base_class_weights = load_json(cw_path)["class_weights"]
BOOST = 1.5   # faktor boost kelas minoritas (tunable; 1.0 = tanpa boost)
ft_class_weights = [(w * BOOST if c in WEAK else w) for c, w in enumerate(base_class_weights)]

print(f"{'kelas':<16}{'baseline':>10}{'fine-tune':>11}")
for c in range(N_CLASSES):
    tag = "  <- boost" if c in WEAK else ""
    print(f"{CLASS_NAMES[c]:<16}{base_class_weights[c]:>10.4f}{ft_class_weights[c]:>11.4f}{tag}")


In [ ]:
# === Build DataLoaders (sampler class-balanced) + model (encoder BEKU) + loss ===
from forestwatch.data.dataset import build_dataloaders_from_files
from forestwatch.model.architecture import build_unet, count_parameters
from forestwatch.model.losses import make_loss_fn
from forestwatch.training.trainer import set_encoder_trainable
import torch

N_WORKERS = 2 if ENV == "colab" else min(8, (os.cpu_count() or 2))

# Cache sampler TERPISAH (FT_SAMPLER) -- WAJIB: compute_patch_sampler_weights cache-hit per-key &
# TIDAK menghitung ulang saat class_weights berubah; pakai cache lama = bobot stale (boost diabaikan).
train_loader, val_loader, _test_loader = build_dataloaders_from_files(
    final_train_files, val_p, test_p,
    batch_size=cfg["training"]["batch_size"],
    num_workers=N_WORKERS,
    augment_p=cfg["training"]["augmentation"],
    class_weights=ft_class_weights,            # -> WeightedRandomSampler (oversample minoritas)
    sampler_cache=FT_SAMPLER, sampler_keys=None,
    persistent_workers=True,
)
print(f"Train {len(train_loader.dataset)} | Val {len(val_loader.dataset)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_unet(in_channels=cfg["model"]["in_channels"], classes=cfg["model"]["classes"],
                   encoder_weights=cfg["model"]["encoder_weights"], **MODEL_ARCH)
model.load_state_dict(torch.load(BASE_CKPT, map_location="cpu"))
model = model.to(device)

# DECOUPLED FINE-TUNE: bekukan encoder (fitur ImageNet generalizable, Kang dkk. 2020).
# decoder + segmentation_head tetap trainable.
assert set_encoder_trainable(model, False), "Gagal bekukan encoder (model tak punya .encoder?)."
n_train = count_parameters(model)
n_total = sum(p.numel() for p in model.parameters())
print(f"Param trainable (decoder+head): {n_train:,} / total {n_total:,} "
      f"({100 * n_train / n_total:.1f}%) -- encoder BEKU")

# Loss: focal_tversky DASAR (recall minoritas via beta>alpha) + CE berbobot supaya boost
# kelas minoritas BENAR-BENAR berpengaruh di loss (preset 'focal_tversky' murni abaikan class_weights).
_lc = cfg["training"]["loss"]
loss_fn = make_loss_fn(
    components=[("focal", 0.4), ("tversky", 0.4), ("ce", 0.2)],
    class_weights=ft_class_weights,
    tversky_alpha=_lc.get("tversky_alpha", 0.3),
    tversky_beta=_lc.get("tversky_beta", 0.7),
    focal_gamma=_lc.get("focal_gamma", 2.0),
    device=device,
)
print("Loss: 0.4*Focal + 0.4*Tversky(beta>alpha) + 0.2*CE(weighted minoritas-boost)")


## Baseline (BEFORE) + tetapkan floor kelas mayoritas (anti-bocor: VAL utk guard, TEST utk laporan)


In [ ]:
# === Baseline (BEFORE) per-kelas IoU: VAL (utk guard) + TEST (utk laporan akhir) ===
# Loop manual baca .npz satu-satu (TANPA DataLoader worker) -> RAM stabil (versi DataLoader
# terbukti bocor puluhan GB di lingkungan ini). VAL menetapkan 'floor' kelas mayoritas;
# TEST HANYA laporan -- JANGAN dipakai utk seleksi checkpoint (cegah kebocoran).
import numpy as np, torch, gc
from forestwatch.training.metrics import compute_confusion_matrix, metric_summary

def _eval_files(mdl, files):
    mdl.eval()
    cm = np.zeros((N_CLASSES, N_CLASSES), dtype=np.int64)
    with torch.no_grad():
        for i, fp in enumerate(files):
            d = np.load(fp)
            img = torch.from_numpy(d["img"]).float().unsqueeze(0).to(device)
            pr = mdl(img).argmax(1).squeeze(0).cpu().numpy().astype(np.uint8)
            cm += compute_confusion_matrix(pr, d["lab"], n_classes=N_CLASSES)
            del d, img, pr
            if i % 3000 == 0:
                gc.collect(); torch.cuda.empty_cache()
    return metric_summary(cm, class_names=CLASS_NAMES)

base_val  = _eval_files(model, val_p)
base_test = _eval_files(model, test_p)
base_val_iou  = [r["iou"] for r in base_val["per_class"]]
base_test_iou = [r["iou"] for r in base_test["per_class"]]

EPS = 0.005   # toleransi penurunan kelas mayoritas yg masih diterima (guard)
floor = {c: base_val_iou[c] - EPS for c in STRONG}
print(f"Baseline VAL mIoU={base_val['mean_iou']:.4f} | TEST mIoU={base_test['mean_iou']:.4f} "
      f"| TEST FWIoU={base_test['fwiou']:.4f}")
print("Floor mayoritas (VAL):", {CLASS_NAMES[c]: round(floor[c], 4) for c in STRONG})
print(f"\n{'kelas':<16}{'VAL IoU':>9}{'TEST IoU':>10}")
for c in range(N_CLASSES):
    print(f"{CLASS_NAMES[c]:<16}{base_val_iou[c]:>9.4f}{base_test_iou[c]:>10.4f}")


## Fine-tune (encoder beku, decoder+head) — seleksi checkpoint terjaga + resume-safe


In [ ]:
# === Fine-tune (encoder BEKU) -- seleksi checkpoint TERJAGA (guarded) ===
# Epoch jadi 'best' HANYA bila IoU kelas mayoritas (VAL) >= floor; di antara yg lolos, ambil
# val mIoU tertinggi -> kelas mayoritas dijamin tak turun. Resume-safe (anti sesi Colab mati).
import time, torch
from torch.amp import autocast
from torchmetrics.classification import MulticlassJaccardIndex
from tqdm.auto import tqdm
from forestwatch.training.metrics import set_seed
try:
    from torch.amp import GradScaler; _NEWSCALER = True      # torch >= 2.4
except ImportError:
    from torch.cuda.amp import GradScaler; _NEWSCALER = False  # torch < 2.4 (Jetson)

FT_LR = 1e-4; FT_EPOCHS = 20; FT_PATIENCE = 8
set_seed(cfg["project"]["seed"])
use_amp = torch.cuda.is_available()

trainable = [p for p in model.parameters() if p.requires_grad]   # decoder + head (encoder beku)
optimizer = torch.optim.AdamW(trainable, lr=FT_LR, weight_decay=cfg["training"]["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FT_EPOCHS)
scaler = (GradScaler(device.type) if _NEWSCALER else GradScaler()) if use_amp else None
iou_pc = MulticlassJaccardIndex(num_classes=N_CLASSES, average=None).to(device)

start_epoch, best_obj, best_epoch, wait, history = 1, -1.0, -1, 0, []
if FT_RESUME.exists():
    try:
        st = torch.load(FT_RESUME, map_location=device)
        model.load_state_dict(st["model"]); optimizer.load_state_dict(st["optimizer"])
        scheduler.load_state_dict(st["scheduler"])
        if scaler and st.get("scaler"): scaler.load_state_dict(st["scaler"])
        start_epoch = st["epoch"] + 1; best_obj = st["best_obj"]; best_epoch = st["best_epoch"]
        wait = st["wait"]; history = st["history"]
        set_encoder_trainable(model, False)   # pastikan encoder tetap beku pasca-resume
        print(f"Resume -> epoch {start_epoch} (best guarded val mIoU={best_obj:.4f})")
    except Exception as e:
        print("Gagal resume:", e)

for ep in range(start_epoch, FT_EPOCHS + 1):
    model.train(); model.encoder.eval()   # encoder BN tetap beku (decoder/head latih)
    tr = 0.0; t0 = time.time()
    for x, y in tqdm(train_loader, desc=f"ep{ep:02d} train", leave=False):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            with autocast(device_type=device.type):
                loss = loss_fn(model(x), y)
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            loss = loss_fn(model(x), y); loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, 1.0); optimizer.step()
        tr += float(loss.item())
    scheduler.step(); tr /= max(len(train_loader), 1)

    model.eval(); iou_pc.reset(); vl = 0.0
    with torch.no_grad():
        for x, y in tqdm(val_loader, desc=f"ep{ep:02d} val", leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            p = model(x); vl += float(loss_fn(p, y).item()); iou_pc.update(p.argmax(1), y)
    vl /= max(len(val_loader), 1)
    pc = [float(v) for v in iou_pc.compute().tolist()]
    vmiou = sum(pc) / len(pc); weak_miou = sum(pc[c] for c in WEAK) / len(WEAK)
    strong_ok = all(pc[c] >= floor[c] for c in STRONG)
    history.append({"epoch": ep, "train_loss": tr, "val_loss": vl, "val_miou": vmiou,
                    "weak_miou": weak_miou, "val_iou_per_class": [round(v, 4) for v in pc],
                    "strong_ok": bool(strong_ok), "lr": optimizer.param_groups[0]["lr"],
                    "epoch_time_sec": time.time() - t0})
    print(f"ep{ep:02d} | loss {tr:.4f} | val {vl:.4f} | mIoU {vmiou:.4f} | "
          f"weak {weak_miou:.4f} | mayoritas_ok={strong_ok}")
    print("   IoU/kelas:", [round(v, 3) for v in pc])

    if strong_ok and vmiou > best_obj:
        best_obj, best_epoch, wait = vmiou, ep, 0
        torch.save(model.state_dict(), FT_CKPT)
        print(f"   -> best guarded checkpoint (val mIoU={vmiou:.4f}) -> {FT_CKPT.name}")
    else:
        wait += 1
    torch.save({"epoch": ep, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict() if scaler else None,
                "best_obj": best_obj, "best_epoch": best_epoch, "wait": wait,
                "history": history}, FT_RESUME)
    if wait >= FT_PATIENCE:
        print(f"Early stopping di epoch {ep} (patience={FT_PATIENCE})."); break

print(f"\nSelesai. best guarded val mIoU={best_obj:.4f} @ epoch {best_epoch}")
if best_epoch < 0:
    print("PERINGATAN: TAK ada epoch yg lolos guard (semua menurunkan kelas mayoritas).")
    print("-> FT_CKPT tidak tersimpan. Pertahankan baseline; coba turunkan BOOST/FT_LR. JANGAN promosikan.")


In [ ]:
# === Plot kurva fine-tune -> FT_DIR ===
import matplotlib.pyplot as plt
assert history, "history kosong -- jalankan cell fine-tune dulu."
eps = [h["epoch"] for h in history]
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
ax[0].plot(eps, [h["train_loss"] for h in history], label="train", lw=2)
ax[0].plot(eps, [h["val_loss"] for h in history], label="val", lw=2)
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(eps, [h["val_miou"] for h in history], color="green", lw=2, label="val mIoU")
ax[1].plot(eps, [h["weak_miou"] for h in history], color="red", lw=2, label="weak mIoU")
ax[1].axhline(base_val["mean_iou"], color="gray", ls="--", label="baseline mIoU")
ax[1].set_title("mIoU (val)"); ax[1].set_ylim(0, 1); ax[1].legend(); ax[1].grid(alpha=0.3)
for c in range(N_CLASSES):
    ax[2].plot(eps, [h["val_iou_per_class"][c] for h in history],
               color=CLASS_COLORS[c], lw=1.6, label=CLASS_NAMES[c])
ax[2].set_title("val IoU per-kelas"); ax[2].set_ylim(0, 1)
ax[2].legend(fontsize=7, ncol=2); ax[2].grid(alpha=0.3)
fig.suptitle(MODEL_KEY + " (fine-tune)"); fig.tight_layout()
fig.savefig(FT_DIR / "training_curve_finetune.png", dpi=120, bbox_inches="tight"); plt.show()
print("Disimpan:", FT_DIR / "training_curve_finetune.png")


## Evaluasi akhir di TEST (sekali) — tabel BEFORE/AFTER + vonis jujur


In [ ]:
# === Evaluasi TEST akhir (sekali) + tabel BEFORE/AFTER + vonis jujur ===
import numpy as np, torch
import matplotlib.pyplot as plt
from forestwatch.model.architecture import build_unet, export_to_onnx

if not FT_CKPT.exists():
    print("FT_CKPT tak ada -> fine-tune tak menghasilkan model yg lolos guard.")
    print("VONIS: pertahankan BASELINE. Tidak ada artefak fine-tune utk dipromosikan.")
else:
    ft_model = build_unet(in_channels=cfg["model"]["in_channels"], classes=cfg["model"]["classes"],
                          encoder_weights=cfg["model"]["encoder_weights"], **MODEL_ARCH).to(device)
    ft_model.load_state_dict(torch.load(FT_CKPT, map_location="cpu"))
    ft_test = _eval_files(ft_model, test_p)
    ft_iou = [r["iou"] for r in ft_test["per_class"]]

    print(f"{'kelas':<16}{'BEFORE':>9}{'AFTER':>9}{'delta':>9}")
    drop_majority = []
    for c in range(N_CLASSES):
        dlt = ft_iou[c] - base_test_iou[c]
        flag = ""
        if c in STRONG and dlt < -EPS:
            flag = "  <- MAYORITAS TURUN"; drop_majority.append(c)
        elif c in WEAK and dlt > 0:
            flag = "  <- naik"
        print(f"{CLASS_NAMES[c]:<16}{base_test_iou[c]:>9.4f}{ft_iou[c]:>9.4f}{dlt:>+9.4f}{flag}")
    print(f"\nmIoU  : {base_test['mean_iou']:.4f} -> {ft_test['mean_iou']:.4f} "
          f"({ft_test['mean_iou'] - base_test['mean_iou']:+.4f})")
    print(f"FWIoU : {base_test['fwiou']:.4f} -> {ft_test['fwiou']:.4f} "
          f"({ft_test['fwiou'] - base_test['fwiou']:+.4f})")

    mi_up = ft_test["mean_iou"] >= base_test["mean_iou"]
    if mi_up and not drop_majority:
        print("\nVONIS: BERHASIL -- mIoU naik & kelas mayoritas tak turun. Layak dipromosikan "
              "(lihat cell promosi, set PROMOTE=True).")
    else:
        why = []
        if not mi_up: why.append("mIoU TIDAK naik di test")
        if drop_majority: why.append("mayoritas turun: " + ", ".join(CLASS_NAMES[c] for c in drop_majority))
        print("\nVONIS: BELUM memenuhi target (" + "; ".join(why) + "). Pertahankan baseline / "
              "tune ulang (BOOST, FT_LR, FT_EPOCHS). JANGAN promosikan.")

    save_json(ft_test, FT_DIR / "metrics_finetune.json")
    save_json({"model_key": MODEL_KEY + "_finetune", **MODEL_ARCH,
               "method": "decoupled cRT (encoder frozen, decoder+head) + focal_tversky+CE(minority-boost)",
               "boost": BOOST, "ft_lr": FT_LR, "best_epoch_guarded": best_epoch,
               "baseline_test_miou": base_test["mean_iou"], "finetune_test_miou": ft_test["mean_iou"],
               "baseline_test_per_class_iou": base_test_iou, "finetune_test_per_class_iou": ft_iou,
               "per_class": ft_test["per_class"]}, FT_DIR / "summary_finetune.json")

    cm = np.array(ft_test["confusion_matrix"]); cmn = cm / cm.sum(axis=1, keepdims=True).clip(1)
    fig, axx = plt.subplots(figsize=(8, 6)); im = axx.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            axx.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=9,
                     color="white" if cmn[i, j] > 0.5 else "black")
    axx.set_xticks(range(N_CLASSES)); axx.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    axx.set_yticks(range(N_CLASSES)); axx.set_yticklabels(CLASS_NAMES)
    axx.set_xlabel("Predicted"); axx.set_ylabel("True"); axx.set_title("Confusion - fine-tune")
    fig.colorbar(im, ax=axx); fig.tight_layout()
    fig.savefig(FT_DIR / "confusion_matrix_finetune.png", dpi=120, bbox_inches="tight"); plt.show()

    try:
        export_to_onnx(ft_model, FT_DIR / "model_finetune.onnx",
                       in_channels=cfg["model"]["in_channels"], patch_size=cfg["inference"]["patch_size"])
        print("ONNX:", FT_DIR / "model_finetune.onnx")
    except Exception as e:
        print("ONNX dilewati:", e)
    print("Disimpan:", FT_DIR / "metrics_finetune.json", "| summary_finetune.json | confusion_matrix_finetune.png")


In [ ]:
# === (Opsional) Promosikan model fine-tune ke lokasi baseline -- HANYA bila VONIS berhasil ===
# Default OFF. Set PROMOTE=True secara SADAR setelah cek tabel BEFORE/AFTER. Baseline di-backup
# dulu (best_model_prefinetune_backup.pt) -> tetap reversible.
PROMOTE = False
import shutil
if not PROMOTE:
    print("PROMOTE=False -- tidak menyalin apa pun. Baseline tetap aktif.")
elif not FT_CKPT.exists():
    print("FT_CKPT tak ada -- tak ada yg dipromosikan.")
elif ENV not in ("colab", "lab"):
    print("Promosi otomatis hanya colab/lab (Drive). Di kaggle, unduh artefak dari:", FT_DIR)
else:
    bak = BASE_DIR / "best_model_prefinetune_backup.pt"
    if not bak.exists():
        shutil.copy(BASE_DIR / "best_model.pt", bak); print("Backup baseline ->", bak)
    shutil.copy(FT_CKPT, BASE_DIR / "best_model.pt")
    shutil.copy(FT_DIR / "metrics_finetune.json", BASE_DIR / "metrics.json")
    if (FT_DIR / "model_finetune.onnx").exists():
        shutil.copy(FT_DIR / "model_finetune.onnx", BASE_DIR / "model.onnx")
    print("Dipromosikan ->", BASE_DIR, "(baseline sudah di-backup; reversible).")
